In [1]:
%matplotlib inline

%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
sys.path.append('..')

In [3]:
import random
import numpy as np
import pylab as plt

from stable_baselines3 import PPO, DQN
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, SubprocVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.monitor import Monitor

In [4]:
from vimms.Common import POSITIVE, set_log_level_warning, load_obj
from vimms.ChemicalSamplers import MZMLFormulaSampler, MZMLRTandIntensitySampler, MZMLChromatogramSampler
from vimms.Noise import UniformSpikeNoise
from vimms.Evaluation import evaluate_real
from mass_spec_utils.data_import.mzmine import load_picked_boxes

from vimms_gym.env import DDAEnv
from vimms_gym.chemicals import generate_chemicals
from vimms_gym.features import obs_to_dfs
from vimms_gym.evaluation import evaluate
from vimms_gym.policy import fullscan_policy, random_policy, topN_policy, best_ppo_policy
from vimms_gym.evaluation import evaluate, run_method

/opt/anaconda3/envs/vimms-gym/lib/python3.9/site-packages/statsmodels/compat/pandas.py:61: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import Int64Index as NumericIndex
/opt/anaconda3/envs/vimms-gym/lib/python3.9/site-packages/psims/mzmlb/writer.py:15: UserWarning: hdf5plugin is missing! Only the slower GZIP compression scheme will be available! Please install hdf5plugin to be able to use Blosc.
  warnings.warn(


# 1. Parameters

In [5]:
# n_chemicals = (200, 500)
# mz_range = (100, 600)
# rt_range = (0, 300)
# intensity_range = (1E4, 1E10)

In [6]:
n_chemicals = (2000, 5000)
mz_range = (100, 600)
rt_range = (200, 1000)
intensity_range = (1E4, 1E10)

In [7]:
min_mz = mz_range[0]
max_mz = mz_range[1]
min_rt = rt_range[0]
max_rt = rt_range[1]
min_log_intensity = np.log(intensity_range[0])
max_log_intensity = np.log(intensity_range[1])

In [8]:
isolation_window = 0.7
N = 10
rt_tol = 15
mz_tol = 10
min_ms1_intensity = 5000
ionisation_mode = POSITIVE

enable_spike_noise = True
noise_density = 0.1
noise_max_val = 1e3

In [ ]:
mzml_filename = 'fullscan_QCB.mzML'
mz_sampler = MZMLFormulaSampler(mzml_filename, min_mz=min_mz, max_mz=max_mz)
ri_sampler = MZMLRTandIntensitySampler(mzml_filename, min_rt=min_rt, max_rt=max_rt,
                                       min_log_intensity=min_log_intensity,
                                       max_log_intensity=max_log_intensity)
cr_sampler = MZMLChromatogramSampler(mzml_filename)

2022-03-31 20:17:56.573 | DEBUG    | mass_spec_utils.data_import.mzml:_load_file:166 - Loaded 2471 scans
2022-03-31 20:17:59.508 | DEBUG    | mass_spec_utils.data_import.mzml:_load_file:166 - Loaded 2471 scans


In [ ]:
params = {
    'chemical_creator': {
        'mz_range': mz_range,
        'rt_range': rt_range,
        'intensity_range': intensity_range,
        'n_chemicals': n_chemicals,
        'mz_sampler': mz_sampler,
        'ri_sampler': ri_sampler,
        'cr_sampler': cr_sampler,
    },
    'noise': {
        'enable_spike_noise': enable_spike_noise,
        'noise_density': noise_density,
        'noise_max_val': noise_max_val,
        'mz_range': mz_range
    },
    'env': {
        'ionisation_mode': ionisation_mode,
        'rt_range': rt_range,
        'isolation_window': isolation_window,
        'mz_tol': mz_tol,
        'rt_tol': rt_tol,
    }
}

In [ ]:
max_peaks = 100
in_dir = 'results_%d_%d' % (max_peaks, rt_tol)

In [ ]:
n_eval_episodes = 10
deterministic = True

# 2. Evaluation

#### Generate some chemical sets

In [ ]:
n_eval_episodes = 1
eval_dir = 'evaluation'
methods = [
    'PPO',
    'TopN',
    'random',    
]

In [ ]:
chemical_creator_params = params['chemical_creator']

chem_list = []
for i in range(n_eval_episodes):
    print(i)
    chems = generate_chemicals(chemical_creator_params)
    chem_list.append(chems)

#### Run different methods

In [ ]:
len(chem_list[0])

In [ ]:
max_peaks

In [ ]:
out_dir = '%s/eval_%d_%d' % (eval_dir, max_peaks, rt_tol)
in_dir, out_dir

#### Compare to Top-10

In [ ]:
copy_params = dict(params)
copy_params['env']['rt_tol'] = rt_tol

for method in methods:
    banner = 'method = %s max_peaks = %d rt_tol = %d' % (method, max_peaks, rt_tol)
    print(banner)
    print()
    
    if method == 'PPO':
        fname = os.path.join(in_dir, 'model_%s.zip' % method)
        model = PPO.load(fname)
    elif method == 'DQN':
        fname = os.path.join(in_dir, 'model_%s.zip' % method)
        model = DQN.load(fname)
    else:
        model = None

    env = DDAEnv(max_peaks, copy_params)
    run_method(env, chem_list, method, out_dir, min_ms1_intensity=min_ms1_intensity, model=model, N=10, print_eval=True)
    print()

#### Test classic Top-N controller in ViMMS

In [ ]:
from vimms.MassSpec import IndependentMassSpectrometer
from vimms.Controller import TopNController
from vimms.Environment import Environment

In [ ]:
chemicals = chem_list[0]
len(chemicals)

In [ ]:
set_log_level_warning()

In [ ]:
noise_params = params['noise']
noise_density = noise_params['noise_density']
noise_max_val = noise_params['noise_max_val']
noise_min_mz = noise_params['mz_range'][0]
noise_max_mz = noise_params['mz_range'][1]
spike_noise = UniformSpikeNoise(noise_density, noise_max_val, min_mz=noise_min_mz,
                                max_mz=noise_max_mz)
mass_spec = IndependentMassSpectrometer(ionisation_mode, chems, spike_noise=spike_noise)

In [ ]:
controller = TopNController(ionisation_mode, N, isolation_window, mz_tol, rt_tol, min_ms1_intensity)
env = Environment(mass_spec, controller, min_rt, max_rt, progress_bar=True, out_dir=out_dir, out_file='TopN_controller.mzML')
env.run()

In [ ]:
evaluate(env)

### Test on QCB data

#### Load pre-processed QCB chemicals

In [ ]:
chem_list = load_obj('datasets_500_0_5000.p')
chemicals = chem_list[0]
len(chemicals)

#### Filter chemicals by mz and RT range

In [ ]:
filtered = []
for chem in chemicals:
    if (min_mz < chem.isotopes[0][0] < max_mz) and (min_rt < chem.rt < max_rt):
        filtered.append(chem)
        
len(filtered)

In [ ]:
filtered_chem_list = [filtered]

#### Disable spike noise in DDAEnv

In [ ]:
copy_params = dict(params)
copy_params['noise']['enable_spike_noise'] = False
copy_params

#### Test different methods

In [ ]:
for method in methods:
    banner = 'method = %s max_peaks = %d rt_tol = %d' % (method, max_peaks, rt_tol)
    print(banner)
    print()
    
    if method == 'PPO':
        fname = os.path.join(in_dir, 'model_%s.zip' % method)
        model = PPO.load(fname)
    elif method == 'DQN':
        fname = os.path.join(in_dir, 'model_%s.zip' % method)
        model = DQN.load(fname)
    else:
        model = None

    env = DDAEnv(max_peaks, copy_params)
    run_method(env, filtered_chem_list, method, out_dir, min_ms1_intensity=min_ms1_intensity, model=model, 
               N=10, mzml_prefix='QCB')
    print()

#### Test classic Top-N controller in ViMMS

In [ ]:
mass_spec = IndependentMassSpectrometer(ionisation_mode, filtered)

In [ ]:
controller = TopNController(ionisation_mode, N, isolation_window, mz_tol, rt_tol, min_ms1_intensity)
env = Environment(mass_spec, controller, min_rt, max_rt, progress_bar=True, out_dir=out_dir, 
                  out_file='QCB_TopN_controller.mzML')
env.run()

#### Evaluation from mzML

In [ ]:
boxes = load_picked_boxes('fullscan_QCB_box.csv')

In [ ]:
mzmls = [
    os.path.join(out_dir, 'PPO_0.mzML'),
    os.path.join(out_dir, 'TopN_0.mzML'),
    os.path.join(out_dir, 'TopN_controller.mzML'),
    os.path.join(out_dir, 'random_0.mzML'),    
]
for mzml in mzmls:
    res = evaluate_real([mzml], boxes)
    cov_prop = res['coverage_prop']
    int_prop = res['intensity_prop']
    print(mzml, cov_prop, int_prop)